# Experimental Stoch backends

These examples make the base measure, numerical integration, partition sequence, and sampling uncertainty explicit. They do not implement the full category `Stoch`.

In [ ]:
import math

import numpy as np

from markov_entropy.divergences import KL
from markov_entropy.stoch import (
    DensityDistribution,
    density_kl,
    density_total_variation,
    estimate_expectation,
    partition_lower_bounds,
)

## User-supplied quadrature on the unit interval

In [ ]:
nodes, weights = np.polynomial.legendre.leggauss(120)
locations = (nodes + 1.0) / 2.0
scaled_weights = weights / 2.0
def integrate(function):
    total = sum(
        w * function(float(x))
        for x, w in zip(locations, scaled_weights, strict=True)
    )
    return float(total)

uniform = DensityDistribution(lambda _: 0.0, 'uniform')
tilted = DensityDistribution(lambda x: math.log(x + 0.5), 'tilted')
kl_value = density_kl(uniform, tilted, integrate)
tv_value = density_total_variation(uniform, tilted, integrate)
assert kl_value > 0 and 0 < tv_value < 1

## Divergence lower bounds from finite partitions

In [ ]:
partition_estimate = partition_lower_bounds(
    [
        ([0.5, 0.5], [0.5, 0.5]),
        ([0.4, 0.1, 0.1, 0.4], [0.25, 0.25, 0.25, 0.25]),
    ],
    KL(),
    require_monotone=True,
)
assert partition_estimate.is_monotone
partition_estimate.values

## Monte Carlo estimates report uncertainty

In [ ]:
rng = np.random.default_rng(20260826)
samples = rng.normal(size=2000)
estimate = estimate_expectation(samples, lambda value: value**2)
assert abs(estimate.value - 1.0) < 0.1
assert estimate.standard_error > 0
estimate